In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
import matplotlib

import matplotlib.pyplot as plt

import time
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
import umap
from sklearn.cluster import DBSCAN
from sklearn import metrics

from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as pl
import shap

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
%load_ext autoreload
%autoreload 2

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/BRCA_SHAP/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
from PermCell_Smooth import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

import xgboost as xgb
import mlx_umap as UMAP
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import pandas as pd
from POgSET import *
import icecream as ic

# Load and initialize

In [ ]:
import glob

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/CyTOF_CR7/CyTOF7_scMCF7/csv_scale_value/"

In [ ]:
FList=glob.glob(dir+"*")

In [ ]:
FList.sort()
FList

In [ ]:
DBs=[f"exp{i:02}" for i in range(1,7)]

In [ ]:
DBs=['MCF7_Yael','MCF7_Uri','MCF7_C5','MCF7_Yael_P','MCF7_C3','MCF7_C4']

In [ ]:
for F,DB in zip(FList,DBs):
    print(F)
    globals()[DB]=pd.read_csv(F,)

In [ ]:
Rep=dict(zip(list(MCF7_Yael.columns),[f.split("_")[-1] for f in list(MCF7_Yael.columns)]))

In [ ]:
Rep

In [ ]:
for DB in DBs:
    globals()[DB].rename(columns=Rep,inplace=True)

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
Rep['H3K27Ac']='H3K27ac'

In [ ]:
for F,DB in zip(FList,DBs):
#    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
#    globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)

In [ ]:
N=list(MCF7_Yael.columns)
N.sort()


In [ ]:
N

In [ ]:
DBs

In [ ]:
Run="MCF7_C4"

In [ ]:
DBs=[Run]

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)

In [ ]:
EpiCols.remove('BMI1')
CellIden.append('BMI1')

In [ ]:
EpiCols.remove('EZH2')
CellIden.append('EZH2')

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
len(NamesAll)

In [ ]:
for DB in DBs:
    plt.figure()
    D=np.arcsinh(globals()[DB]/5).copy()
    sns.histplot(data=D,x='H3',**hKWD,color='blue')
    sns.histplot(data=D,x='H3.3',**hKWD,color='red')
    sns.histplot(data=D,x='H4',**hKWD,color='magenta')
#    sns.histplot(data=D,x='H2A',**hKWD,color='g')
    plt.title(DB)
#plt.xscale('log')
#plt.yscale('log')

## Gate on H3.3/H2A too low, but also remove outliers 99.99% from all 

In [ ]:
GateColumns=['H3.3','H4','H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
#    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data




In [ ]:
for DB in DBs:
    globals()[DB]=Gate(globals()[DB],DB)



# Normalize using new method on all intercellular markers

In [ ]:
def R(p,x,data,Q,M,M1,M2,M3):
    a=p['a']
    b=p['b']
    d=x.divide(a*M1+(1-a-b)*M2+b*M3,axis=0)
    return d.std()['H3.3']**2+d.std()['H4']**2+d.std()['H3']**2

def NormalizeNew(data):
    
    params = Parameters()
    params.add('a', value=0.1,min=0,max=1)
    params.add('b', value=0.1,min=0,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4','H3']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
    M3=(ddf/Q)['H3']

    out=minimize(R, params ,args=(ddf, ddf,Q,M,M1,M2,M3),method='cg')
    AA=out.params['a'].value
    BB=out.params['b'].value
    M=M1*AA+M2*(1-AA-BB)+M3*BB
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf
    ddf2[NormMRK]=data[NormMRK]
    data=ddf2.copy()
    print(data.shape,ddf2.shape)

    del ddf 
    del ddf2
    return data
    
def R2(p,x,data,Q,M,M1,M2):
    a=p['a']
    d=x.divide(a*M1+(1-a)*M2,axis=0)
    return (d.std()['H3.3'])**2+(d.std()['H4'])**2

def NormalizeNew2(data):
    
    params = Parameters()
    params.add('a', value=0.5,min=0.1,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf[NormMRK].mean()
    M=(ddf/Q)[['H3.3','H4']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
 
    out=minimize(R2, params ,args=(ddf[NormMRK], ddf[NormMRK],Q,M,M1,M2),method='cg')
    AA=out.params['a'].value
    print(AA)
    M=M1*AA+M2*(1-AA)
    ddf[NormMRK]=ddf[NormMRK].divide(M,axis=0).copy()
    data=ddf.copy()
    print(data.shape,ddf2.shape)
    ddf2[NormMRK]=data[NormMRK]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data

In [ ]:
for DB in DBs:
    globals()[DB]=NormalizeNew(globals()[DB])



In [ ]:
scFac=5
for DB in DBs:
    globals()[DB]=np.arcsinh(globals()[DB]/scFac)


In [ ]:
MRK_All=NamesAll.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')
#MRK_All.remove('H2A')

EPC=EpiCols.copy()
Core=['H3','H3.3','H4']#,'H2A']
for C in Core:
    EPC.remove(C)

In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
    globals()[DB]['Line']=DB


params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")

In [ ]:
import distinctipy

In [ ]:
DBCLR=dict(zip(DBs,distinctipy.get_colors(6)))

In [ ]:
a=[]
fig, axs = plt.subplots(11, 3, figsize=(20, 20))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)
        
for i,N in enumerate(MRK_All):
    print(N)
    for DB in DBs:
        
        sns.histplot(data=globals()[DB],x=N,ax=a[i],**hKWD,color=DBCLR[DB],label=f'{DB}')

    
#    a[i].set_title(N)

plt.subplots_adjust(wspace=0.5, hspace=1.9)
a[0].legend()    

#fig.savefig("Plots/AllRaw.pdf",dpi=200,bbox_inches='tight')

In [ ]:
pKWD={'dpi':200,'bbox_inches':'tight'}

In [ ]:
DBs

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB]]).copy()


In [ ]:
Mat=CAll.groupby('Line').mean()

In [ ]:
Mat=Mat.loc[DBs,:]

In [ ]:
NC=5000
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB].sample(NC,replace=False)]).copy()
                  

In [ ]:
import mlx_umap.dropin as UMAP
UM=UMAP(min_dist=0.001,n_neighbors=15,random_state=42,verbose=True)
from mlx_umap import UMAP
UM = UMAP(min_dist=0.001, n_neighbors=15, random_state=42, verbose=True)
X_2d=UM.fit_transform(CAll[EPC])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
AD=ad.AnnData(CAll[MRK_All],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
AD.write_h5ad(f"Data/AD_{Run}.h5ad")

In [ ]:
DBs

In [ ]:
from collections import Counter

In [ ]:
lbl=DBSCAN(eps=0.25,min_samples=30).fit_predict(X_2d)
Counter(lbl)

In [ ]:
lbl[lbl==7]=5
lbl[lbl==10]=5
lbl[lbl==1]=5

lbl[lbl==9]=4
lbl[lbl==11]=4
lbl[lbl==3]=4
lbl[lbl==8]=4

In [ ]:
del AD.uns['Cl_DB_colors']

In [ ]:
AD.obs['Cl_DB']=lbl
AD.obs['Cl_DB']=AD.obs['Cl_DB'].astype('category')

In [ ]:
M=lbl>-1
AD=AD[M]

In [ ]:
sc.pl.umap(AD,color=['Cl_DB'],cmap='seismic',vmin='p01',vmax='p99',show=False);

In [ ]:
sc.pl.umap(AD,color=MRK_All+['Line','Cl_DB'],cmap='seismic',vmin='p01',vmax='p99',show=False);
#plt.savefig(f"Plots/UMAP_{Run}.png",dpi=200,bbox_inches='tight')

In [ ]:
marker_sets = {
    # Luminal/epithelial program: epithelial & ER axis up; mesenchymal/basal down
    "Epithelial_Luminal": {
        "up":   {"EpCAM", "E-cadherin", "ER", "GATA3", "KRT8-18"},# "Pan-KRT",
        "down": {"KRT5", "Vimentin", "aSMA"}
    },

    "Basal_Noa": {
        "up":   {"H3K4me1", "H3K4me3","H3K9me2"},
        "down": {"H4K20me3","H3K36me3"}
    },

    "Proliferation": {
        "up":   {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},
        "down": set()  
    },
}




In [ ]:
Z, P, Zabs, Pabs, Zdir = sipsic_like_scores_v3(
    AD, marker_sets,
    n_perm=2048,                   # or even 16 for smoke test
    prefer_permutation=True,     # <- important
    perm_batch=512,              # keeps RAM flat
    normalize_set_weights="l2",
    use_sparse_W=False,
    progress=True               # progress bars can add overhead in some envs
)

In [ ]:
ADUS=ad.AnnData(obs=Z)
ADUS.obsm['X_umap']=AD.obsm['X_umap']
#ADUS.obs['Class']=AD.obs['Class'].values
sc.pl.umap(ADUS,color=list(ADUS.obs.columns),cmap='magma_r',show=False,vcenter=0,)

In [ ]:
DF=pd.concat([AD.to_df(),AD.obs],axis=1)


In [ ]:
DF.groupby('Cl_DB').size()/DF.shape[0]

In [ ]:
X_2d=AD.obsm['X_umap']

In [ ]:
DF.to_parquet(f"Data/{Run}_MRK.parquet")
ADUS.obs.to_parquet(f"Data/{Run}_Sigs.parquet")
pd.DataFrame(X_2d,columns=['umap1','umap2']).to_parquet(f"Data/{Run}_UMAP.parquet")

In [ ]:
DF.shape,ADUS.obs.shape,X_2d.shape

In [ ]:
CAll=CAll[M]

In [ ]:
CAll['Cl']=AD.obs['Cl_DB'].astype(int).values

In [ ]:
Mat=CAll.groupby(['Cl']).mean(numeric_only=True)

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(Mat[EPC].T,cmap='seismic',center=0,annot=True,annot_kws={'size':8})
#plt.savefig(f"Plots/{Run}_HM_Cl.png",dpi=200,bbox_inches='tight')

In [ ]:
MRK=list(set(NamesAll).difference(set(EpiCols)))

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(Mat[MRK].T,cmap='seismic',center=0,annot=True,annot_kws={'size':8})
#plt.savefig(f"Plots/{Run}_HM_Cl_2.png",dpi=200,bbox_inches='tight')

In [ ]:
AD.write_h5ad("/Users/ronguy/Temp/Test.h5ad")

In [ ]:
for DB in DBs:
    M=AD.obs['Line']==DB
    sc.pl.umap(AD[M],color=MRK_All,cmap='seismic',vmin='p01',vmax='p99',show=False);
#    #plt.savefig(f"Plots/{DB}_UMAP_Epi.png",dpi=200,bbox_inches='tight')

In [ ]:
X_2d=UM.fit_transform(CAll[CellIden])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
AD=ad.AnnData(CAll[MRK_All],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
sc.pl.umap(AD,color=MRK_All+['Line'],cmap='seismic',vmin='p01',vmax='p99',show=False);
#plt.savefig(f"Plots/UMAP_{Run}_CellIden.png",dpi=200,bbox_inches='tight')